# Notebook 14: vLLM Internals

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 05 (KV Cache), 07 (PagedAttention), 11 (Continuous Batching)

---

## Background: Why vLLM Matters

In June 2023, a team from UC Berkeley published "Efficient Memory Management for Large Language Model Serving with PagedAttention" and released vLLM. Within months it became the de facto standard for open-source LLM serving, and most major cloud providers (Google Cloud, AWS, Azure) use it or concepts directly derived from it internally.

The headline number: **vLLM achieves 10–24× higher throughput than Hugging Face's `pipeline()`** on the same hardware, with zero quality degradation. This wasn't from a better model or better hardware — it was purely from a better systems architecture.

To understand *why*, you need to understand how vLLM solves the three core problems of LLM serving:

### Problem 1: KV Cache Fragmentation

When you pre-allocate KV cache memory for requests, you don't know the output length in advance. So you either:
- Over-allocate (waste memory for short completions)
- Under-allocate (fail with OOM for long completions)

Before PagedAttention, every serving system allocated a *contiguous* block of memory for each request's KV cache — like allocating an entire page of RAM for a 1-byte variable. Fragmentation was severe: 60-80% of GPU memory was wasted.

**vLLM's solution**: PagedAttention. KV cache is allocated in fixed-size blocks (like OS virtual memory pages), scattered through GPU memory, connected by block tables. No pre-allocation, no fragmentation.

### Problem 2: Static Batching Waste

Traditional batching locks a GPU until every request in the batch finishes. Short requests sit idle waiting for long ones.

**vLLM's solution**: Continuous batching (iteration-level scheduling). At every decode step, evict finished requests and admit new ones. The GPU is always busy.

### Problem 3: Memory Pressure from Growing Sequences

As each request generates tokens, its KV cache grows. The scheduler must balance:
- Admitting new requests (needs free blocks)
- Keeping in-progress requests alive (holds blocks)
- Preempting requests when memory is tight (releases blocks, accepts recompute cost)

**vLLM's solution**: A BlockManager that tracks free/allocated blocks and implements swap-out or recompute preemption.

### The Architecture

vLLM has three main components that work together:

```
┌─────────────────────────────────────────────┐
│                LLMEngine                      │
│  ┌──────────────┐   ┌────────────────────┐   │
│  │  Scheduler   │   │   BlockManager     │   │
│  │  (what runs) │──▶│  (KV block alloc)  │   │
│  └──────────────┘   └────────────────────┘   │
│           │                                   │
│           ▼                                   │
│  ┌──────────────────────────────────────┐    │
│  │         Worker (GPU process)         │    │
│  │  ModelRunner → PagedAttention kernel │    │
│  └──────────────────────────────────────┘    │
└─────────────────────────────────────────────┘
```

In this notebook, we'll build a simplified but faithful simulation of each component,
then wire them together into a working vLLM-style engine.

---

## What You'll Build

1. **SequenceGroup** — the data structure representing a request and its generation state
2. **BlockManager** — KV block allocation, free lists, block tables
3. **Scheduler** — prefill/decode prioritization, admission control, preemption
4. **LLMEngine** — the main loop wiring everything together
5. **End-to-end trace** — visualize what the engine does step by step

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
from collections import deque
import random

random.seed(42)
np.random.seed(42)
print("Setup complete.")

## Part 1: Request State Machine

In vLLM, each request progresses through a well-defined state machine.
Understanding these states is the key to understanding the scheduler.

In [ ]:
class SequenceStatus(Enum):
    """
    State machine for a request sequence in vLLM.
    
    WAITING → RUNNING: scheduler admits the request
    RUNNING → WAITING: scheduler preempts (memory pressure)
    RUNNING → FINISHED: EOS token or max_tokens reached
    RUNNING → SWAPPED: preempted to CPU memory
    SWAPPED → WAITING: swapped back in (waiting for GPU slot)
    """
    WAITING  = 'waiting'   # In queue, not yet started
    RUNNING  = 'running'   # In active batch
    SWAPPED  = 'swapped'   # KV cache on CPU RAM
    FINISHED = 'finished'  # Generation complete


@dataclass
class LogicalBlock:
    """A logical block slot in a sequence's block table."""
    block_number: int       # logical position in this sequence's KV cache
    physical_block: Optional[int] = None  # which physical GPU block holds this data
    ref_count: int = 0      # for copy-on-write (prefix sharing)


@dataclass
class SequenceData:
    """The token-level data for a sequence."""
    prompt_token_ids: list[int]
    output_token_ids: list[int] = field(default_factory=list)
    
    @property
    def length(self) -> int:
        return len(self.prompt_token_ids) + len(self.output_token_ids)
    
    def add_token(self, token_id: int) -> None:
        self.output_token_ids.append(token_id)


@dataclass
class SequenceGroup:
    """
    The central data structure in vLLM.
    
    A SequenceGroup represents one user request. In beam search,
    it would hold multiple Sequences (one per beam). Here we use
    a single sequence per group for simplicity.
    """
    group_id: int
    data: SequenceData
    max_output_tokens: int
    arrival_time: int = 0
    status: SequenceStatus = SequenceStatus.WAITING
    
    # Block table: maps logical block index → physical block id
    block_table: list[int] = field(default_factory=list)
    
    # Timing
    start_time: Optional[int] = None
    finish_time: Optional[int] = None
    preempt_count: int = 0
    
    @property
    def is_prefill(self) -> bool:
        """True if this request hasn't generated any output yet."""
        return len(self.data.output_token_ids) == 0
    
    @property
    def is_finished(self) -> bool:
        return len(self.data.output_token_ids) >= self.max_output_tokens
    
    @property
    def n_tokens(self) -> int:
        return self.data.length
    
    @property
    def n_blocks_needed(self) -> int:
        """Number of KV blocks needed given current sequence length."""
        return (self.n_tokens + BLOCK_SIZE - 1) // BLOCK_SIZE


BLOCK_SIZE = 16  # tokens per KV block (vLLM default is 16)

# Example sequence group
sg = SequenceGroup(
    group_id=0,
    data=SequenceData(prompt_token_ids=list(range(45))),  # 45-token prompt
    max_output_tokens=100,
)
print(f"Sequence group: {sg.n_tokens} tokens, needs {sg.n_blocks_needed} blocks")
print(f"  Block size: {BLOCK_SIZE} tokens")
print(f"  KV memory: {sg.n_blocks_needed} × {BLOCK_SIZE} = {sg.n_blocks_needed * BLOCK_SIZE} token slots")

## Part 2: BlockManager

The BlockManager is the memory allocator for KV cache. It maintains a free list of
physical blocks and maps logical block indices (per sequence) to physical locations.

In [ ]:
class BlockManager:
    """
    KV cache block allocator — the core of PagedAttention.
    
    Manages GPU blocks (and optionally CPU blocks for swap).
    Each physical block holds KV states for BLOCK_SIZE tokens.
    """
    def __init__(self, num_gpu_blocks: int, num_cpu_blocks: int = 100):
        self.num_gpu_blocks = num_gpu_blocks
        self.num_cpu_blocks = num_cpu_blocks
        
        # Free block lists (using sets for O(1) operations)
        self.free_gpu_blocks: set[int] = set(range(num_gpu_blocks))
        self.free_cpu_blocks: set[int] = set(range(num_cpu_blocks))
        
        # Reference counting for copy-on-write (prefix caching)
        self.gpu_block_refs: dict[int, int] = {}
        
        # Stats
        self.n_allocations = 0
        self.n_frees = 0
        self.n_swaps = 0
    
    @property
    def num_free_gpu_blocks(self) -> int:
        return len(self.free_gpu_blocks)
    
    @property
    def gpu_utilization(self) -> float:
        return 1.0 - len(self.free_gpu_blocks) / self.num_gpu_blocks
    
    def can_allocate(self, seq_group: SequenceGroup) -> bool:
        """Check if we have enough free blocks for this request."""
        needed = seq_group.n_blocks_needed - len(seq_group.block_table)
        return needed <= self.num_free_gpu_blocks
    
    def allocate(self, seq_group: SequenceGroup) -> None:
        """Allocate GPU blocks for a new or growing sequence."""
        needed = seq_group.n_blocks_needed - len(seq_group.block_table)
        if needed <= 0:
            return
        if needed > self.num_free_gpu_blocks:
            raise RuntimeError(f"OOM: need {needed} blocks, have {self.num_free_gpu_blocks}")
        
        for _ in range(needed):
            block_id = self.free_gpu_blocks.pop()
            seq_group.block_table.append(block_id)
            self.gpu_block_refs[block_id] = 1
            self.n_allocations += 1
    
    def free(self, seq_group: SequenceGroup) -> None:
        """Free all GPU blocks held by this sequence group."""
        for block_id in seq_group.block_table:
            self.gpu_block_refs[block_id] = self.gpu_block_refs.get(block_id, 1) - 1
            if self.gpu_block_refs.get(block_id, 0) <= 0:
                self.free_gpu_blocks.add(block_id)
                self.n_frees += 1
        seq_group.block_table.clear()
    
    def can_append_slot(self, seq_group: SequenceGroup) -> bool:
        """Check if we can grow this sequence by one more block (if needed)."""
        current_blocks = len(seq_group.block_table)
        needed = seq_group.n_blocks_needed
        if needed > current_blocks:
            return self.num_free_gpu_blocks >= 1
        return True  # No new block needed yet
    
    def append_slot(self, seq_group: SequenceGroup) -> Optional[int]:
        """Allocate a new block if the sequence has grown into a new block."""
        current_blocks = len(seq_group.block_table)
        needed = seq_group.n_blocks_needed
        if needed > current_blocks:
            block_id = self.free_gpu_blocks.pop()
            seq_group.block_table.append(block_id)
            self.gpu_block_refs[block_id] = 1
            self.n_allocations += 1
            return block_id
        return None
    
    def swap_out(self, seq_group: SequenceGroup) -> dict[int, int]:
        """Move a sequence's blocks from GPU to CPU (for preemption)."""
        gpu_to_cpu = {}
        new_block_table = []
        
        for gpu_block in seq_group.block_table:
            if self.free_cpu_blocks:
                cpu_block = self.free_cpu_blocks.pop()
                gpu_to_cpu[gpu_block] = cpu_block
                self.free_gpu_blocks.add(gpu_block)
                new_block_table.append(cpu_block)
                self.n_swaps += 1
        
        seq_group.block_table = new_block_table
        return gpu_to_cpu
    
    def swap_in(self, seq_group: SequenceGroup) -> dict[int, int]:
        """Move a swapped-out sequence back from CPU to GPU."""
        cpu_to_gpu = {}
        new_block_table = []
        
        for cpu_block in seq_group.block_table:
            if self.free_gpu_blocks:
                gpu_block = self.free_gpu_blocks.pop()
                cpu_to_gpu[cpu_block] = gpu_block
                self.free_cpu_blocks.add(cpu_block)
                new_block_table.append(gpu_block)
        
        seq_group.block_table = new_block_table
        return cpu_to_gpu


# Demo: allocate blocks for a few sequences
bm = BlockManager(num_gpu_blocks=20)

seqs = [
    SequenceGroup(i, SequenceData(list(range(30 + i * 15))), max_output_tokens=50)
    for i in range(4)
]

print(f"Total GPU blocks: {bm.num_gpu_blocks}")
for seq in seqs:
    bm.allocate(seq)
    print(f"  Seq {seq.group_id}: {seq.n_tokens} tokens → {seq.n_blocks_needed} blocks "
          f"→ block_table: {seq.block_table}")

print(f"\nAfter allocation: {bm.num_free_gpu_blocks} free blocks, {bm.gpu_utilization:.0%} utilization")

# Free one sequence
bm.free(seqs[0])
print(f"After freeing seq 0: {bm.num_free_gpu_blocks} free blocks")

## Part 3: Scheduler

The Scheduler is the brain of vLLM. Every iteration it decides:
1. Which waiting requests to admit (start prefill)
2. Which running requests to continue (decode step)
3. Which running requests to preempt if memory is tight

In [ ]:
@dataclass
class SchedulerOutput:
    """What the scheduler decided to do this step."""
    prefill_groups: list[SequenceGroup] = field(default_factory=list)   # start prefill
    decode_groups: list[SequenceGroup] = field(default_factory=list)    # continue decode
    preempted_groups: list[SequenceGroup] = field(default_factory=list) # preempted
    swapped_in_groups: list[SequenceGroup] = field(default_factory=list) # resumed from swap
    
    @property
    def n_batched_tokens(self) -> int:
        prefill_tokens = sum(g.n_tokens for g in self.prefill_groups)
        decode_tokens = len(self.decode_groups)  # one token per decode group
        return prefill_tokens + decode_tokens


class Scheduler:
    """
    vLLM-style scheduler: iteration-level scheduling with preemption.
    
    Policy:
    - Running sequences get highest priority (don't preempt without reason)
    - New sequences (prefill) are admitted if memory allows
    - When memory is tight: preempt the *last-admitted* running sequence
      (LIFO preemption minimizes total wasted compute)
    """
    def __init__(
        self,
        block_manager: BlockManager,
        max_num_seqs: int = 256,
        max_num_batched_tokens: int = 4096,
        preemption_mode: str = 'recompute',  # 'recompute' or 'swap'
    ):
        self.block_manager = bm = block_manager
        self.max_num_seqs = max_num_seqs
        self.max_num_batched_tokens = max_num_batched_tokens
        self.preemption_mode = preemption_mode
        
        self.waiting: deque[SequenceGroup] = deque()   # FIFO queue of new requests
        self.running: list[SequenceGroup] = []          # currently active
        self.swapped: list[SequenceGroup] = []          # swapped to CPU
    
    def add_request(self, seq_group: SequenceGroup) -> None:
        """Accept a new request into the waiting queue."""
        self.waiting.append(seq_group)
    
    def schedule(self, current_step: int) -> SchedulerOutput:
        """
        The main scheduling loop. Called once per decode iteration.
        Returns what to run this step.
        """
        output = SchedulerOutput()
        
        # ── Phase 1: Grow running sequences (append slots for new tokens) ──
        preempted = []
        for seq in list(self.running):
            if seq.is_finished:
                seq.status = SequenceStatus.FINISHED
                seq.finish_time = current_step
                self.block_manager.free(seq)
                self.running.remove(seq)
                continue
            
            if not self.block_manager.can_append_slot(seq):
                # Memory tight — preempt this sequence
                self.running.remove(seq)
                seq.preempt_count += 1
                
                if self.preemption_mode == 'swap':
                    self.block_manager.swap_out(seq)
                    seq.status = SequenceStatus.SWAPPED
                    self.swapped.append(seq)
                else:
                    # Recompute: discard blocks, put back in waiting queue
                    self.block_manager.free(seq)
                    seq.data.output_token_ids.clear()  # start over
                    seq.status = SequenceStatus.WAITING
                    self.waiting.appendleft(seq)  # front of queue (high priority)
                
                preempted.append(seq)
            else:
                self.block_manager.append_slot(seq)
                output.decode_groups.append(seq)
        
        output.preempted_groups = preempted
        
        # ── Phase 2: Swap in previously swapped sequences (if memory allows) ──
        for seq in list(self.swapped):
            if self.block_manager.num_free_gpu_blocks >= len(seq.block_table):
                self.block_manager.swap_in(seq)
                seq.status = SequenceStatus.RUNNING
                self.swapped.remove(seq)
                self.running.append(seq)
                output.swapped_in_groups.append(seq)
                output.decode_groups.append(seq)
        
        # ── Phase 3: Admit new requests from waiting queue ──
        while self.waiting and len(self.running) < self.max_num_seqs:
            seq = self.waiting[0]
            
            if not self.block_manager.can_allocate(seq):
                break  # Memory full — stop admitting
            
            # Check token budget
            pending_tokens = output.n_batched_tokens + seq.n_tokens
            if pending_tokens > self.max_num_batched_tokens and output.prefill_groups:
                break  # Batch token budget exceeded
            
            self.waiting.popleft()
            self.block_manager.allocate(seq)
            seq.status = SequenceStatus.RUNNING
            seq.start_time = current_step
            self.running.append(seq)
            output.prefill_groups.append(seq)
        
        return output


# Demo: schedule a few requests
bm2 = BlockManager(num_gpu_blocks=30)
sched = Scheduler(bm2, max_num_seqs=8)

# Add 5 requests
for i in range(5):
    sg = SequenceGroup(
        group_id=i,
        data=SequenceData(list(range(20 + i * 10))),
        max_output_tokens=30 + i * 5,
        arrival_time=0,
    )
    sched.add_request(sg)

out = sched.schedule(current_step=0)
print(f"Step 0 schedule:")
print(f"  Prefill: {len(out.prefill_groups)} requests ({[g.group_id for g in out.prefill_groups]})")
print(f"  Decode:  {len(out.decode_groups)} requests")
print(f"  Total batched tokens: {out.n_batched_tokens}")
print(f"  Free GPU blocks: {bm2.num_free_gpu_blocks}/{bm2.num_gpu_blocks}")

## Part 4: LLMEngine — The Main Loop

The LLMEngine wires together the Scheduler, BlockManager, and a mock "Worker" (which in
real vLLM runs the actual model forward pass on the GPU).

In [ ]:
class MockWorker:
    """
    Simulates the GPU worker that runs the model.
    In real vLLM this calls the actual attention kernel + sampling.
    Here we just pick random tokens and pretend.
    """
    def __init__(self, vocab_size: int = 32000, eos_token_id: int = 2):
        self.vocab_size = vocab_size
        self.eos_token_id = eos_token_id
        self.total_forward_passes = 0
    
    def execute_model(
        self,
        prefill_groups: list[SequenceGroup],
        decode_groups: list[SequenceGroup],
    ) -> dict[int, int]:  # group_id → next_token
        """
        One forward pass: processes all prefill and decode inputs together.
        Returns next token for each sequence.
        
        In real vLLM:
        - Prefill tokens are packed together in one contiguous batch
        - Decode tokens are one per sequence, attended over their full KV cache
        - PagedAttention kernel handles the scattered KV blocks transparently
        """
        self.total_forward_passes += 1
        outputs = {}
        
        for group in prefill_groups + decode_groups:
            # Sample a random token (occasionally EOS to end sequences)
            if random.random() < 0.03:  # 3% chance of EOS each step
                token = self.eos_token_id
            else:
                token = random.randint(3, self.vocab_size - 1)
            outputs[group.group_id] = token
        
        return outputs


@dataclass
class EngineStats:
    total_steps: int = 0
    total_tokens_generated: int = 0
    total_preemptions: int = 0
    total_swaps: int = 0
    gpu_utilization_history: list[float] = field(default_factory=list)
    batch_size_history: list[int] = field(default_factory=list)
    finished_groups: list[SequenceGroup] = field(default_factory=list)


class LLMEngine:
    """
    Simplified vLLM-style engine.
    
    Wires together: RequestQueue → Scheduler → Worker → Output.
    The core loop: schedule → execute_model → update_state → collect_outputs.
    """
    def __init__(
        self,
        num_gpu_blocks: int = 200,
        max_num_seqs: int = 32,
        max_batched_tokens: int = 4096,
        block_size: int = 16,
        preemption_mode: str = 'recompute',
    ):
        global BLOCK_SIZE
        BLOCK_SIZE = block_size
        
        self.block_manager = BlockManager(num_gpu_blocks)
        self.scheduler = Scheduler(self.block_manager, max_num_seqs, max_batched_tokens, preemption_mode)
        self.worker = MockWorker()
        self.stats = EngineStats()
        self.current_step = 0
    
    def add_request(self, seq_group: SequenceGroup) -> None:
        self.scheduler.add_request(seq_group)
    
    def step(self) -> list[SequenceGroup]:
        """
        One engine step:
        1. Scheduler decides what to run
        2. Worker executes one forward pass
        3. Update sequence states with new tokens
        4. Return finished sequences
        """
        sched_out = self.scheduler.schedule(self.current_step)
        
        if not sched_out.prefill_groups and not sched_out.decode_groups:
            return []
        
        # Execute model (one forward pass, all groups together)
        next_tokens = self.worker.execute_model(
            sched_out.prefill_groups, sched_out.decode_groups
        )
        
        # Update sequence states
        newly_finished = []
        all_active = sched_out.prefill_groups + sched_out.decode_groups
        for group in all_active:
            token = next_tokens.get(group.group_id)
            if token is not None:
                group.data.add_token(token)
                self.stats.total_tokens_generated += 1
            
            if group.is_finished or token == self.worker.eos_token_id:
                group.status = SequenceStatus.FINISHED
                group.finish_time = self.current_step
                newly_finished.append(group)
        
        # Stats
        self.stats.total_preemptions += len(sched_out.preempted_groups)
        self.stats.gpu_utilization_history.append(self.block_manager.gpu_utilization)
        self.stats.batch_size_history.append(len(all_active))
        self.stats.total_steps += 1
        self.stats.finished_groups.extend(newly_finished)
        
        self.current_step += 1
        return newly_finished
    
    def has_work(self) -> bool:
        return bool(self.scheduler.waiting or self.scheduler.running or self.scheduler.swapped)


# Run a full simulation
engine = LLMEngine(num_gpu_blocks=200, max_num_seqs=16)

rng = np.random.RandomState(42)
N_REQUESTS = 80
for i in range(N_REQUESTS):
    sg = SequenceGroup(
        group_id=i,
        data=SequenceData(list(range(rng.randint(20, 200)))),
        max_output_tokens=rng.randint(20, 200),
        arrival_time=0,
    )
    engine.add_request(sg)

print("Running LLMEngine simulation...")
max_steps = 5000
step = 0
while engine.has_work() and step < max_steps:
    engine.step()
    step += 1

print(f"\nSimulation complete:")
print(f"  Steps:              {engine.stats.total_steps}")
print(f"  Tokens generated:   {engine.stats.total_tokens_generated}")
print(f"  Preemptions:        {engine.stats.total_preemptions}")
print(f"  Finished requests:  {len(engine.stats.finished_groups)}")
print(f"  Avg GPU utilization:{np.mean(engine.stats.gpu_utilization_history):.1%}")
print(f"  Avg batch size:     {np.mean(engine.stats.batch_size_history):.1f}")
print(f"  Throughput:         {engine.stats.total_tokens_generated / engine.stats.total_steps:.1f} tok/step")

## Part 5: Visualizing the Engine State

Let's visualize what the engine is doing over time:
- GPU block utilization
- Batch composition (prefill vs decode)
- Preemption events

In [ ]:
# Run a detailed simulation with event logging
@dataclass
class StepEvent:
    step: int
    n_prefill: int
    n_decode: int
    n_preempted: int
    n_finished: int
    gpu_utilization: float
    free_blocks: int


def run_engine_with_events(
    n_requests: int = 50,
    num_gpu_blocks: int = 150,
    max_num_seqs: int = 12,
) -> list[StepEvent]:
    engine2 = LLMEngine(num_gpu_blocks=num_gpu_blocks, max_num_seqs=max_num_seqs)
    rng = np.random.RandomState(99)
    
    for i in range(n_requests):
        sg = SequenceGroup(
            group_id=i,
            data=SequenceData(list(range(rng.randint(10, 150)))),
            max_output_tokens=int(np.clip(rng.lognormal(3.5, 1.2), 10, 200)),
        )
        engine2.add_request(sg)
    
    events = []
    max_steps = 3000
    
    for s in range(max_steps):
        if not engine2.has_work():
            break
        
        # Peek at scheduler state before stepping
        sched_out = engine2.scheduler.schedule(engine2.current_step)
        n_prefill = len(sched_out.prefill_groups)
        n_decode = len(sched_out.decode_groups)
        n_preempted = len(sched_out.preempted_groups)
        
        # Actually execute
        next_tokens = engine2.worker.execute_model(sched_out.prefill_groups, sched_out.decode_groups)
        finished = []
        for group in sched_out.prefill_groups + sched_out.decode_groups:
            token = next_tokens.get(group.group_id)
            if token:
                group.data.add_token(token)
            if group.is_finished:
                group.finish_time = s
                finished.append(group)
        
        events.append(StepEvent(
            step=s,
            n_prefill=n_prefill,
            n_decode=n_decode,
            n_preempted=n_preempted,
            n_finished=len(finished),
            gpu_utilization=engine2.block_manager.gpu_utilization,
            free_blocks=engine2.block_manager.num_free_gpu_blocks,
        ))
        engine2.current_step += 1
    
    return events


events = run_engine_with_events(n_requests=50)
steps = [e.step for e in events]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# GPU utilization
ax = axes[0]
gpu_utils = [e.gpu_utilization for e in events]
ax.fill_between(steps, gpu_utils, alpha=0.6, color='#3498db')
ax.plot(steps, gpu_utils, color='#2980b9', linewidth=0.8)
ax.set_ylabel("GPU Block Utilization")
ax.set_title("vLLM Engine State Over Time")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))
ax.grid(alpha=0.3)

# Batch composition (prefill vs decode)
ax2 = axes[1]
n_prefills = [e.n_prefill for e in events]
n_decodes = [e.n_decode for e in events]
ax2.stackplot(steps, n_decodes, n_prefills,
              labels=['Decode', 'Prefill'],
              colors=['#2ecc71', '#e67e22'], alpha=0.8)
ax2.set_ylabel("Active Requests")
ax2.legend(loc='upper right')
ax2.grid(alpha=0.3)

# Preemptions and finishes
ax3 = axes[2]
preempt_steps = [e.step for e in events if e.n_preempted > 0]
finish_steps = [e.step for e in events if e.n_finished > 0]
ax3.vlines(finish_steps, 0, 1, color='#2ecc71', alpha=0.5, linewidth=1, label='Request finished')
ax3.vlines(preempt_steps, 0, 1, color='#e74c3c', alpha=0.8, linewidth=1.5, label='Preemption')
ax3.set_ylabel("Events")
ax3.set_xlabel("Step")
ax3.legend()
ax3.set_yticks([])
ax3.grid(alpha=0.3)

import matplotlib.ticker as ticker
plt.tight_layout()
plt.savefig('vllm_engine_state.png', dpi=120, bbox_inches='tight')
plt.show()

n_preemptions = sum(e.n_preempted for e in events)
print(f"Total preemption events: {n_preemptions}")
print(f"Avg GPU utilization: {np.mean(gpu_utils):.1%}")

## Part 6: vLLM in Practice

How to actually use vLLM for serving. The Python API and server mode.

In [ ]:
# vLLM usage patterns (requires: pip install vllm)

vllm_usage = '''
# ─── Option 1: Python API (offline batch inference) ───────────────────────────
from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    tensor_parallel_size=1,  # number of GPUs
    dtype="bfloat16",
    max_model_len=4096,      # context window
    gpu_memory_utilization=0.90,  # fraction of GPU VRAM to use for KV cache
)

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=512,
)

prompts = ["Tell me about attention mechanisms.", "Write a Python hello world."]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(output.outputs[0].text)


# ─── Option 2: OpenAI-compatible server ───────────────────────────────────────
# Start server:
#   vllm serve meta-llama/Llama-3.1-8B-Instruct --port 8000

# Then call like any OpenAI endpoint:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="token")

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
    stream=True,
)
for chunk in response:
    print(chunk.choices[0].delta.content, end="", flush=True)


# ─── Key configuration knobs ──────────────────────────────────────────────────
# gpu_memory_utilization: 0.90 → 90% of VRAM for KV cache (leave 10% for model)
# max_num_seqs: max concurrent requests in flight
# block_size: KV block size (16 default; try 32 for longer contexts)
# enable_prefix_caching: reuse KV cache for shared prefixes across requests
# speculative_model: enable speculative decoding with a draft model
# tensor_parallel_size: number of GPUs for tensor parallelism
# pipeline_parallel_size: number of GPUs for pipeline parallelism
'''

print(vllm_usage)

# Configuration guide
print("vLLM Configuration Cheat Sheet:")
print("=" * 70)
configs = [
    ("gpu_memory_utilization", "0.85-0.95", "Higher = more KV cache = more concurrent requests"),
    ("max_num_seqs",           "32-512",    "Max concurrent requests; bounded by KV memory"),
    ("block_size",             "16 or 32",  "KV tokens per block; 32 helps long sequences"),
    ("enable_prefix_caching", "True",      "Cache shared prefixes (system prompts)"),
    ("speculative_model",      "7B draft",  "Use draft model for 2-3x decode speedup"),
    ("tensor_parallel_size",   "n_gpus",    "Shard model across GPUs (needed for 70B+)"),
    ("max_model_len",          "model max","Limit context to save KV memory"),
    ("dtype",                  "bfloat16",  "Use FP8 for A100/H100 for extra speed"),
]
for param, default, desc in configs:
    print(f"  {param:<30} {default:<12} {desc}")

## Summary

vLLM achieved 10–24× throughput improvement not through better models or faster hardware,
but through solving the systems problems that previous serving frameworks ignored.

### Key Takeaways

**Three Core Problems, Three Solutions**  
- **KV fragmentation** → PagedAttention (scattered KV blocks, no pre-allocation)
- **Static batching waste** → Continuous batching (iteration-level scheduling)
- **Memory pressure** → BlockManager with swap/recompute preemption

**The Data Structures**  
- `SequenceGroup`: a request + its generation state + block table
- `BlockManager`: free list of physical blocks, allocation/free/swap operations
- `Scheduler`: FCFS + LIFO preemption + max_num_seqs limit

**The Main Loop**  
Every iteration: schedule → execute one forward pass → update states → emit outputs.
The forward pass handles both prefill tokens (variable length) and decode tokens (one per
active request) in a single batch — this is what makes it efficient.

**Using vLLM**  
Python API for batch inference; OpenAI-compatible server for production serving.
Key knobs: `gpu_memory_utilization` (KV cache budget), `max_num_seqs` (concurrency),
`enable_prefix_caching` (system prompt reuse), `speculative_model` (decode speedup).

### What's Next (Notebook 15: Structured Decoding)

Notebook 15 covers structured decoding: how to constrain LLM outputs to valid JSON,
specific formats, or other schemas. This is how tools like Instructor, Outlines, and
guidance work under the hood — and it's essential for building reliable LLM-powered applications.